In [2]:
# --- PASSO 1: EXPLORANDO OS DADOS ---
# Importando as bibliotecas essenciais para análise de dados
import pandas as pd
import numpy as np

# Configurando o Pandas para mostrar todas as colunas
pd.set_option('display.max_columns', None)

# Caminho relativo para o nosso arquivo de dados brutos
caminho_arquivo = '../dados/raw/leads_insightos_raw.csv'

# Carregando o dataset
df = pd.read_csv(caminho_arquivo)

# Visualizando as 5 primeiras linhas para entender os dados
display(df.head())

print("-" * 50)

# Verificando as informações gerais e a quantidade de dados nulos 
df.info()

,id_lead,origem_canal,campanha_tipo,dispositivo,metodo_tracking,paginas_vistas,tempo_sessao_segundos,scroll_depth_max,baixou_material_rico,cargo,tamanho_empresa,tipo_email,converteu
0,1,Meta Ads,Retargeting,Mobile,Server-side,11,451,75%,1,Coordenador,200+,Genérico,0
1,2,Referral,Retargeting,Mobile,Server-side,10,400,75%,0,Coordenador,1-10,Corporativo,1
2,3,Organic Search,Nenhuma,Mobile,Client-side,13,611,100%,0,Gerente,200+,Corporativo,1
3,4,Meta Ads,Lookalike,Mobile,Server-side,6,516,75%,0,C-Level,51-200,Corporativo,1
4,5,Google Ads,Retargeting,Mobile,Server-side,3,270,75%,1,Coordenador,200+,Genérico,1


--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lead                5000 non-null   int64
 1   origem_canal           4141 non-null   str  
 2   campanha_tipo          4141 non-null   str  
 3   dispositivo            5000 non-null   str  
 4   metodo_tracking        5000 non-null   str  
 5   paginas_vistas         5000 non-null   int64
 6   tempo_sessao_segundos  5000 non-null   int64
 7   scroll_depth_max       5000 non-null   str  
 8   baixou_material_rico   5000 non-null   int64
 9   cargo                  5000 non-null   str  
 10  tamanho_empresa        5000 non-null   str  
 11  tipo_email             5000 non-null   str  
 12  converteu              5000 non-null   int64
dtypes: int64(5), str(8)
memory usage: 507.9 KB


In [3]:
# --- PASSO 2: TRATAND OS DADOS NULOS E LIMPANDO A BASE ---
# 1. Preenchendo os buracos deixados pelo rastreamento Client-side
df['origem_canal'] = df['origem_canal'].fillna('Desconhecido/Dark Social')
df['campanha_tipo'] = df['campanha_tipo'].fillna('Sem Rastreio')

# 2. Removendo a coluna de ID, pois ela não tem valor preditivo para o modelo
df = df.drop('id_lead', axis=1)

# 3. Confirmando se a limpeza funcionou
print("Quantidade de valores nulos por coluna após o tratamento:")
display(df.isnull().sum())


Quantidade de valores nulos por coluna após o tratamento:


origem_canal             0
campanha_tipo            0
dispositivo              0
metodo_tracking          0
paginas_vistas           0
tempo_sessao_segundos    0
scroll_depth_max         0
baixou_material_rico     0
cargo                    0
tamanho_empresa          0
tipo_email               0
converteu                0
dtype: int64

In [4]:
# --- PASSO 3: ENCODING (TRADUZINDO TEXTO PARA NÚMEROS) ---

# 1. Variáveis Binárias (Tudo que tem apenas duas opções vira 0 ou 1)
df['dispositivo'] = df['dispositivo'].map({'Desktop': 1, 'Mobile': 0})
df['tipo_email'] = df['tipo_email'].map({'Corporativo': 1, 'Genérico': 0})
df['metodo_tracking'] = df['metodo_tracking'].map({'Server-side': 1, 'Client-side': 0})

# 2. Variáveis Ordinais (Têm uma hierarquia de importância)
mapa_cargos = {'Estagiário': 1, 'Analista': 2, 'Coordenador': 3, 'Gerente': 4, 'C-Level': 5}
df['cargo'] = df['cargo'].map(mapa_cargos)

mapa_tamanho = {'1-10': 1, '11-50': 2, '51-200': 3, '200+': 4}
df['tamanho_empresa'] = df['tamanho_empresa'].map(mapa_tamanho)

# 3. Tratando o Scroll Depth (Removendo o símbolo de '%' e transformando em número inteiro)
df['scroll_depth_max'] = df['scroll_depth_max'].str.replace('%', '').astype(int)

# 4. Variáveis Nominais (One-Hot Encoding para Origens e Campanhas)
df = pd.get_dummies(df, columns=['origem_canal', 'campanha_tipo'], drop_first=True)

# Convertendo os booleanos (True/False) gerados pelo get_dummies para inteiros (1/0)
for col in df.columns:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

# Visualizando a "mágica": como a base ficou após todas as transformações
display(df.head())

,dispositivo,metodo_tracking,paginas_vistas,tempo_sessao_segundos,scroll_depth_max,baixou_material_rico,cargo,tamanho_empresa,tipo_email,converteu,origem_canal_Direct,origem_canal_Google Ads,origem_canal_LinkedIn Ads,origem_canal_Meta Ads,origem_canal_Organic Search,origem_canal_Referral,campanha_tipo_Lookalike,campanha_tipo_Nenhuma,campanha_tipo_PMax,campanha_tipo_Retargeting,campanha_tipo_Search,campanha_tipo_Sem Rastreio
0,0,1,11,451,75,1,3,4,0,0,0,0,0,1,0,0,0,0,0,1,0,0
1,0,1,10,400,75,0,3,1,1,1,0,0,0,0,0,1,0,0,0,1,0,0
2,0,0,13,611,100,0,4,4,1,1,0,0,0,0,1,0,0,1,0,0,0,0
3,0,1,6,516,75,0,5,3,1,1,0,0,0,1,0,0,1,0,0,0,0,0
4,0,1,3,270,75,1,3,4,0,1,0,1,0,0,0,0,0,0,0,1,0,0


In [5]:
# --- PASSO 4: NORMALIZAÇÃO DAS VARIÁVEIS NUMÉRICAS ---

from sklearn.preprocessing import MinMaxScaler

# 1. Selecionando as colunas que têm números em escalas muito grandes/diferentes
colunas_numericas = ['paginas_vistas', 'tempo_sessao_segundos', 'scroll_depth_max']

# 2. Inicializando o "Normalizador"
scaler = MinMaxScaler()

# 3. Aplicando a transformação nos nossos dados
df[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])

# 4. Salvando a nossa base final limpa e processada na pasta correta!
caminho_salvamento = '../dados/processed/leads_insightos_limpo.csv'
df.to_csv(caminho_salvamento, index=False)

# Visualizando o resultado final e confirmando a exportação
print(f"✅ Base de dados limpa e normalizada salva com sucesso em: {caminho_salvamento}")
print("-" * 50)
display(df.head())

✅ Base de dados limpa e normalizada salva com sucesso em: ../dados/processed/leads_insightos_limpo.csv
--------------------------------------------------


,dispositivo,metodo_tracking,paginas_vistas,tempo_sessao_segundos,scroll_depth_max,baixou_material_rico,cargo,tamanho_empresa,tipo_email,converteu,origem_canal_Direct,origem_canal_Google Ads,origem_canal_LinkedIn Ads,origem_canal_Meta Ads,origem_canal_Organic Search,origem_canal_Referral,campanha_tipo_Lookalike,campanha_tipo_Nenhuma,campanha_tipo_PMax,campanha_tipo_Retargeting,campanha_tipo_Search,campanha_tipo_Sem Rastreio
0,0,1,0.769231,0.264093,0.666667,1,3,4,0,0,0,0,0,1,0,0,0,0,0,1,0,0
1,0,1,0.692308,0.232843,0.666667,0,3,1,1,1,0,0,0,0,0,1,0,0,0,1,0,0
2,0,0,0.923077,0.362132,1.000000,0,4,4,1,1,0,0,0,0,1,0,0,1,0,0,0,0
3,0,1,0.384615,0.303922,0.666667,0,5,3,1,1,0,0,0,1,0,0,1,0,0,0,0,0
4,0,1,0.153846,0.153186,0.666667,1,3,4,0,1,0,1,0,0,0,0,0,0,0,1,0,0
